# Interpretability of the Final Model with SHAP

Global interpretability analysis of the Gradient Boosting model using SHAP values.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import os

plt.style.use("seaborn-v0_8")

# Ensure reports directory exists
os.makedirs("../reports", exist_ok=True)

shap.__version__

## Loading Training Data and Final Model

In [ ]:
X_train = pd.read_parquet("../data/dataset/X_train.parquet")
y_train = pd.read_parquet("../data/dataset/y_train.parquet").squeeze()

X_train.shape, y_train.shape, y_train.value_counts(normalize=True)

In [ ]:
final_model = joblib.load("../models/final_diabetes_model.pkl")
final_model

# The model is stored inside a dictionary
real_model = final_model["model"]

real_model

## SHAP Explainer

In [ ]:
# Sample to speed up SHAP computation
X_train_sample = X_train.sample(n=min(2000, len(X_train)), random_state=42)

# SHAP requires a callable → use predict_proba
explainer = shap.Explainer(
    real_model.predict_proba,
    X_train_sample
)

# Compute SHAP values
shap_values = explainer(X_train_sample)

# For binary classification → positive class (index 1)
shap_values_pos = shap_values.values[:, :, 1]

shap_values_pos.shape

## Global Importance (SHAP Bar Plot)

In [ ]:
shap.summary_plot(
    shap_values_pos,
    X_train_sample,
    plot_type="bar",
    show=False
)

plt.title("Global Feature Importance (SHAP)")
plt.tight_layout()
plt.savefig("../reports/SHAP_global_importance.png", dpi=300)
plt.show()

## SHAP Summary Plot (Beeswarm)

In [ ]:
shap.summary_plot(
    shap_values_pos,
    X_train_sample,
    show=False
)

plt.title("SHAP Summary Plot - Final Model")
plt.tight_layout()
plt.savefig("../reports/SHAP_summary_plot.png", dpi=300)
plt.show()